# Milestone 1 lab, notebook 1 of 8: The question and five green checks

## Where we are

| Notebook | What you do |
|---|---|
| **1. The question and five green checks** (you are here) | Meet Lotus Retails and the Monday question. Run the given pipeline for one night. Read its five checks |
| 2. Airflow runs it | Let Airflow run the same night. Read the DAG: its tasks and its arrows |
| 3. The number by hand | Add up one store's receipts yourself. Find out why the two numbers differ |
| 4. Who is hurt | For each table: who uses it, and what happens to them when it is wrong |
| 5. The check card | Meet the page you fill in before any check. Say what one row means |
| 6. Read the table first | Count the rows, find the smallest and largest values, count the empty cells |
| 7. Six questions for any table | Sort your worries under six questions. See which ones the five checks answer |
| 8. The production version | The same logic as a tested program |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`. The [README](../README.md#set-up) installs Docker click by click.

**About ten minutes.**

In [1]:
from datetime import date

from lab_helpers import check_setup, run_pipeline, show_doc, show_file, sql, stores_send

check_setup()

The warehouse is running: Postgres on port 5450.


## Step 1: The business, and the question

**Lotus Retails** is a chain of supermarkets: six stores in three towns. Customers pay at the tills.

Every store manager gets a weekly bonus. It is **1% of the store's net sales**. Net sales are the money taken for sales, less the money given back for refunds.

One person signs for that bonus every Monday. This is what she asks.

In [2]:
show_doc("BRD.md", "2. The question to answer")

**From `docs/BRD.md`, section 2. The question to answer**

Every Monday the head of retail operations signs off the bonus. She asks:

> Can I pay the bonus from last week's net sales, store by store? And if I cannot trust a number, which one, and why?

Today nobody can answer the second sentence. This milestone shows why. It does not fix it yet.

Keep the second sentence in mind: *which one, and why?* The whole project is an answer to it.

## Step 2: The stores send their files

It is the night after **Monday 24 August 2026**. Every store closes its tills and sends its files.

In this project the stores are a small program. The next line makes them send everything up to tonight: four earlier weeks, and this Monday.

In [3]:
stores_send("2026-08-24")

The stores sent 622 files, up to the night of 2026-08-24.
They are in data/landing


Here is one of those files: the sales of store `S03` for Monday. These are its first five rows.

In [4]:
show_file("sales_lines", "sales_lines_S03_2026-08-24.csv")

,line_id,receipt_id,line_no,store_id,till_id,sold_at,product_id,quantity,unit_price_cents,amount_cents,line_type,refund_of_receipt_id,member_id
0,S03-0824-000001,S03-T2-004485,1,S03,T2,2026-08-24 08:05:47,P-038,2.000,399,798,sale,,M-0023
1,S03-0824-000002,S03-T2-004485,2,S03,T2,2026-08-24 08:05:47,P-003,2.000,349,698,sale,,M-0023
2,S03-0824-000003,S03-T1-004491,1,S03,T1,2026-08-24 08:14:30,P-032,-1.000,59,-59,refund,S03-T2-004470,
3,S03-0824-000004,S03-T1-004492,1,S03,T1,2026-08-24 08:39:28,P-026,1.000,259,259,sale,,M-0016
4,S03-0824-000005,S03-T1-004492,2,S03,T1,2026-08-24 08:39:28,P-040,2.000,899,1798,sale,,M-0016


One row is **one item line on one receipt**. The first two rows have the same `receipt_id`, so one customer bought two things.

`amount_cents` is money in whole cents: `798` is $7.98.

## Step 3: Run the pipeline for that night

You do not build this pipeline. It is given to you, ready-made. It has four steps and a report:

1. **load**: copy the files into the warehouse, as received.
2. **clean**: give every column its type, and join the tables that belong together.
3. **check**: run five checks.
4. **publish**: build the two tables people read.

Run it.

In [5]:
run_pipeline("2026-08-24")

load:    622 new files, 37,710 rows copied to the raw tables.
clean:   7 clean tables rebuilt. sales_lines has 20,482 rows.
check:   the five standard checks
  1. The table is not empty           clean.sales_lines    PASS
  2. The key is not null              clean.sales_lines    PASS
  3. The key is unique                clean.sales_lines    PASS
  4. The status is an allowed value   clean.sales_lines    PASS
  5. The total is positive            clean.sales_lines    PASS
  5 of 5 checks passed.
publish: daily_store_sales has 174 rows. replenishment_list has 234 rows for tomorrow.

Lotus Retails. Store managers' bonus, week of Mon 2026-08-24
Net sales from Mon 2026-08-24 to Mon 2026-08-24

  Store                           Manager              Net sales  Bonus (1%)
  S01  Lotus Ashby High Street    Marta Lindqvist        $795.24       $7.95
  S02  Lotus Ashby Market Square  Dana Okafor            $901.28       $9.01
  S03  Lotus Calder Bridge        Tomas Reyes            $796.18     

Read what it printed, from the top.

- It loaded 622 files and built seven clean tables.
- **Five of five checks passed.**
- It printed the bonus report. Store `S03` is at **$796.18** for Monday.

## Step 4: What is a check?

**A check is a query that looks for bad rows.** If the query returns no rows, the data passes.

Here is check 3, "the key is unique". The key of this table is `line_id`. The query looks for a `line_id` that is there more than once.

In [6]:
from lotus_pipeline.checks import STANDARD_CHECKS

key_is_unique = STANDARD_CHECKS[2]
print(key_is_unique.sql)


        SELECT line_id, count(*) AS times
        FROM clean.sales_lines
        GROUP BY line_id
        HAVING count(*) > 1
        


In [7]:
bad_rows = sql(key_is_unique.sql)
print(len(bad_rows), "bad rows. The check passes.")

0 bad rows. The check passes.


A check that never fails teaches nothing. So see it fail once.

Here is the same query on three hand-made lines, where `A-2` is there twice.

In [8]:
sql("""
    SELECT line_id, count(*) AS times
    FROM (VALUES ('A-1'), ('A-2'), ('A-2')) AS three_lines(line_id)
    GROUP BY line_id
    HAVING count(*) > 1
""")

,line_id,times
0,A-2,2


One row comes back. That row is the bad one, so the check fails.

## Step 5: All five

These are the five standard checks. Most teams write them first, on every table, from a template.

In [9]:
night = date(2026, 8, 24)

for check in STANDARD_CHECKS:
    bad_rows = sql(check.sql, (night,) if check.for_the_day else ())
    print(f"{check.number}. {check.name:<32} {len(bad_rows)} bad rows")

1. The table is not empty           0 bad rows
2. The key is not null              0 bad rows
3. The key is unique                0 bad rows
4. The status is an allowed value   0 bad rows


5. The total is positive            0 bad rows


## Step 6: Is that enough to trust the number?

**Data quality** means the data is good enough to trust.

Lotus trusts $796.18 because five checks are green. Now ask a harder question: *what does each of those checks protect?* Who would be hurt if check 4 failed? Nobody at Lotus can say. The checks came from a template.

In notebook 3 you test the number itself.

## What you now know

- The pipeline is given: load, clean, check, publish. You will not build it.
- A check is a query that returns the bad rows. No rows means it passes.
- The five standard checks all pass for the night of 24 August.
- Green checks are a reason to trust a number only if you know what they look at.

**Say it in an interview:** "A check is a query that returns the rows that break a rule. At Lotus Retails the pipeline had five standard checks, all green every night, and nobody could say what any of them protected."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), sections "A check is a query" and "What is good data?".

Next: [notebook 2](02_airflow_runs_it.ipynb). Airflow runs the same night.